# PaddleOCR 파인튜닝 가이드

운송장 이미지 OCR을 위한 PaddleOCR 모델 파인튜닝

## 목차
1. 환경 설정
2. 데이터 준비 (PaddleOCR 형식 변환)
3. 설정 파일 생성
4. 모델 학습
5. 모델 평가 및 추론

## 1. 환경 설정

In [1]:
# PaddlePaddle 및 PaddleOCR 설치
# GPU 버전 (CUDA 11.x)
!pip install paddlepaddle-gpu -i https://pypi.tuna.tsinghua.edu.cn/simple

# CPU 버전 (GPU가 없는 경우)
# !pip install paddlepaddle -i https://pypi.tuna.tsinghua.edu.cn/simple

# PaddleOCR 설치
!pip install paddleocr

Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Defaulting to user installation because normal site-packages is not writeable


In [2]:
# GPU 메모리 최적화 설정 (V100 32GB 전체 사용)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '7'  # GPU 7번 사용
os.environ['FLAGS_fraction_of_gpu_memory_to_use'] = '0.95'  # GPU 메모리 95% 사용
os.environ['FLAGS_eager_delete_tensor_gb'] = '0.0'  # 텐서 즉시 삭제
os.environ['FLAGS_memory_fraction_of_eager_deletion'] = '1.0'  # 메모리 즉시 해제

import paddle
paddle.device.set_device('gpu:0')

# GPU 정보 확인
print(f"PaddlePaddle 버전: {paddle.__version__}")
print(f"GPU 사용 가능: {paddle.is_compiled_with_cuda()}")
print(f"사용 중인 GPU: {paddle.device.get_device()}")

# GPU 메모리 정보
if paddle.is_compiled_with_cuda():
    gpu_props = paddle.device.cuda.get_device_properties()
    print(f"GPU 이름: {gpu_props.name}")
    print(f"GPU 총 메모리: {gpu_props.total_memory / 1024**3:.1f} GB")

PaddlePaddle 버전: 2.6.1
GPU 사용 가능: True
사용 중인 GPU: gpu:0
GPU 이름: Tesla V100-PCIE-32GB
GPU 총 메모리: 31.7 GB


In [3]:
# PaddleOCR 소스 클론 (파인튜닝에 필요)
!git clone https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR
!pip install -r requirements.txt 

fatal: destination path 'PaddleOCR' already exists and is not an empty directory.
/home/j-i14a403/S14P11A403/ai/PaddleOCR
Defaulting to user installation because normal site-packages is not writeable


In [4]:
import os
import json
import shutil
from pathlib import Path
from PIL import Image
import random

# 경로 설정
BASE_DIR = Path('../')  # ai 폴더
GENERATED_DIR = BASE_DIR / 'generated'
PADDLE_DATA_DIR = BASE_DIR / 'paddle_data'

print(f"생성된 이미지 폴더: {GENERATED_DIR}")
print(f"PaddleOCR 데이터 폴더: {PADDLE_DATA_DIR}")

생성된 이미지 폴더: ../generated
PaddleOCR 데이터 폴더: ../paddle_data


## 2. 데이터 준비 - PaddleOCR 형식 변환

PaddleOCR은 두 가지 태스크를 위한 데이터 형식이 있습니다:
- **텍스트 감지 (Detection)**: 이미지에서 텍스트 영역 찾기
- **텍스트 인식 (Recognition)**: 크롭된 텍스트 이미지를 문자로 변환

우리 데이터는 필드별 바운딩 박스가 있으므로 **텍스트 인식** 모델을 파인튜닝합니다.

In [5]:
def convert_to_paddle_recognition_format(generated_dir: Path, output_dir: Path, train_ratio: float = 0.9):
    """
    생성된 데이터를 PaddleOCR 텍스트 인식 형식으로 변환
    
    PaddleOCR Recognition 형식:
    - 이미지: 텍스트 영역만 크롭된 이미지
    - 라벨: image_path\tlabel 형식의 txt 파일
    """
    
    # 출력 디렉토리 생성
    train_img_dir = output_dir / 'train' / 'images'
    val_img_dir = output_dir / 'val' / 'images'
    train_img_dir.mkdir(parents=True, exist_ok=True)
    val_img_dir.mkdir(parents=True, exist_ok=True)
    
    # 라벨 파일 로드
    labels_file = generated_dir / 'labels' / 'labels.json'
    with open(labels_file, 'r', encoding='utf-8') as f:
        all_labels = json.load(f)
    
    # 데이터 섞기
    random.shuffle(all_labels)
    
    # Train/Val 분할
    split_idx = int(len(all_labels) * train_ratio)
    train_labels = all_labels[:split_idx]
    val_labels = all_labels[split_idx:]
    
    train_records = []
    val_records = []
    
    def process_labels(labels, img_dir, records, prefix):
        for idx, label_data in enumerate(labels):
            # 원본 이미지 로드
            img_path = generated_dir / label_data['image_path']
            if not img_path.exists():
                continue
            
            img = Image.open(img_path)
            
            # 각 필드별로 크롭하여 저장
            for field_idx, field in enumerate(label_data['fields']):
                bbox = field['bbox']  # [x1, y1, x2, y2]
                text = field['text']
                field_name = field['field_name']
                
                # 이미지 크롭
                cropped = img.crop((bbox[0], bbox[1], bbox[2], bbox[3]))
                
                # 파일명 생성
                crop_filename = f"{prefix}_{idx:05d}_{field_name}.jpg"
                crop_path = img_dir / crop_filename
                
                # 저장
                cropped.save(crop_path, 'JPEG', quality=95)
                
                # 레코드 추가 (상대 경로 사용)
                relative_path = f"images/{crop_filename}"
                records.append(f"{relative_path}\t{text}")
            
            if (idx + 1) % 100 == 0:
                print(f"{prefix}: {idx + 1}/{len(labels)} 처리 완료")
    
    print("Train 데이터 처리 중...")
    process_labels(train_labels, train_img_dir, train_records, 'train')
    
    print("\nVal 데이터 처리 중...")
    process_labels(val_labels, val_img_dir, val_records, 'val')
    
    # 라벨 파일 저장
    with open(output_dir / 'train' / 'label.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(train_records))
    
    with open(output_dir / 'val' / 'label.txt', 'w', encoding='utf-8') as f:
        f.write('\n'.join(val_records))
    
    print(f"\n변환 완료!")
    print(f"Train 샘플 수: {len(train_records)}")
    print(f"Val 샘플 수: {len(val_records)}")
    
    return len(train_records), len(val_records)

In [6]:
# 데이터 변환 실행
train_count, val_count = convert_to_paddle_recognition_format(
    GENERATED_DIR, 
    PADDLE_DATA_DIR,
    train_ratio=0.9
)

print(f"\nTrain 이미지: {train_count}개")
print(f"Val 이미지: {val_count}개")

Train 데이터 처리 중...
train: 100/27000 처리 완료
train: 200/27000 처리 완료
train: 300/27000 처리 완료
train: 400/27000 처리 완료
train: 500/27000 처리 완료
train: 600/27000 처리 완료
train: 700/27000 처리 완료
train: 800/27000 처리 완료
train: 900/27000 처리 완료
train: 1000/27000 처리 완료
train: 1100/27000 처리 완료
train: 1200/27000 처리 완료
train: 1300/27000 처리 완료
train: 1400/27000 처리 완료
train: 1500/27000 처리 완료
train: 1600/27000 처리 완료
train: 1700/27000 처리 완료
train: 1800/27000 처리 완료
train: 1900/27000 처리 완료
train: 2000/27000 처리 완료
train: 2100/27000 처리 완료
train: 2200/27000 처리 완료
train: 2300/27000 처리 완료
train: 2400/27000 처리 완료
train: 2500/27000 처리 완료
train: 2600/27000 처리 완료
train: 2700/27000 처리 완료
train: 2800/27000 처리 완료
train: 2900/27000 처리 완료
train: 3000/27000 처리 완료
train: 3100/27000 처리 완료
train: 3200/27000 처리 완료
train: 3300/27000 처리 완료
train: 3400/27000 처리 완료
train: 3500/27000 처리 완료
train: 3600/27000 처리 완료
train: 3700/27000 처리 완료
train: 3800/27000 처리 완료
train: 3900/27000 처리 완료
train: 4000/27000 처리 완료
train: 4100/27000 처리 완료
train: 

In [7]:
# 변환된 데이터 확인
print("=" * 50)
print("Train 라벨 샘플:")
print("=" * 50)
with open(PADDLE_DATA_DIR / 'train' / 'label.txt', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        print(line.strip())

Train 라벨 샘플:
images/train_00000_tracking_number.jpg	08104605
images/train_00000_region_code.jpg	서귀포7
images/train_00000_recipient_name.jpg	강옥자
images/train_00000_recipient_address.jpg	서울특별시 마포구 구문로 417
images/train_00000_sender_name.jpg	박경수
images/train_00000_sender_address.jpg	경기도 수원시 영통구 역전대로 340
images/train_00001_tracking_number.jpg	91466966
images/train_00001_region_code.jpg	영암6
images/train_00001_recipient_name.jpg	나하은
images/train_00001_recipient_address.jpg	울산광역시 중구 남문길 173


## 3. 한글 문자 사전 생성

PaddleOCR은 인식할 문자 목록이 필요합니다.

In [21]:
def create_korean_dict(data_dir: Path, output_file: Path):
    """
    학습 데이터에서 사용된 모든 문자를 추출하여 사전 생성
    """
    chars = set()
    
    # Train 라벨에서 문자 추출
    label_file = data_dir / 'train' / 'label.txt'
    with open(label_file, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 2:
                text = parts[1]
                chars.update(text)
    
    # 기본 한글 완성형 추가 (가-힣)
    for code in range(0xAC00, 0xD7A4):
        chars.add(chr(code))
    # 숫자, 영문, 특수문자 추가
    chars.update('0123456789')
    chars.update('abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ')
    chars.update(' .-_,()[]{}:;/\\@#$%&*+=<>?!"\'')
    #chars.update(' -(),')
    # 정렬 후 저장
    sorted_chars = sorted(chars)
    
    with open(output_file, 'w', encoding='utf-8') as f:
        for char in sorted_chars:
            f.write(char + '\n')
    
    print(f"문자 사전 생성 완료: {len(sorted_chars)}개 문자")
    print(f"저장 위치: {output_file}")
    
    return sorted_chars

In [22]:
# 문자 사전 생성
dict_file = PADDLE_DATA_DIR / 'korean_dict.txt'
chars = create_korean_dict(PADDLE_DATA_DIR, dict_file)

문자 사전 생성 완료: 11263개 문자
저장 위치: ../paddle_data/korean_dict.txt


## 4. 설정 파일 생성

PaddleOCR 학습을 위한 YAML 설정 파일을 생성합니다.

In [23]:
# 설정 파일 생성
config_content = f'''
Global:
  debug: false
  use_gpu: true
  epoch_num: 100
  log_smooth_window: 20
  print_batch_step: 10
  save_model_dir: ./output/rec_korean_finetune
  save_epoch_step: 10
  eval_batch_step: [0, 500]
  cal_metric_during_train: true
  pretrained_model: ./pretrain_models/korean_PP-OCRv3_rec_train/best_accuracy
  checkpoints:
  save_inference_dir:
  use_visualdl: false
  infer_img: 
  character_dict_path: {str(PADDLE_DATA_DIR / 'korean_dict.txt').replace(chr(92), '/')}
  max_text_length: 50
  infer_mode: false
  use_space_char: true
  distributed: true
  save_res_path: ./output/rec/predicts.txt

Optimizer:
  name: Adam
  beta1: 0.9
  beta2: 0.999
  lr:
    name: Cosine
    learning_rate: 0.0005
    warmup_epoch: 5
  regularizer:
    name: L2
    factor: 3.0e-05

Architecture:
  model_type: rec
  algorithm: SVTR_LCNet
  Transform:
  Backbone:
    name: MobileNetV1Enhance
    scale: 0.5
    last_conv_stride: [1, 2]
    last_pool_type: avg
  Head:
    name: MultiHead
    head_list:
      - CTCHead:
          Neck:
            name: svtr
            dims: 64
            depth: 2
            hidden_dims: 120
            use_guide: True
          Head:
            fc_decay: 0.00001
      - SARHead:
          enc_dim: 512
          max_text_length: 50

Loss:
  name: MultiLoss
  loss_config_list:
    - CTCLoss:
    - SARLoss:

PostProcess:  
  name: CTCLabelDecode

Metric:
  name: RecMetric
  main_indicator: acc
  ignore_space: False

Train:
  dataset:
    name: SimpleDataSet
    data_dir: {str(PADDLE_DATA_DIR / 'train').replace(chr(92), '/')}
    ext_op_transform_idx: 1
    label_file_list:
      - {str(PADDLE_DATA_DIR / 'train' / 'label.txt').replace(chr(92), '/')}
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - RecConAug:
          prob: 0.5
          ext_data_num: 2
          image_shape: [48, 320, 3]
      - RecAug:
      - MultiLabelEncode:
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys:
            - image
            - label_ctc
            - label_sar
            - length
            - valid_ratio
  loader:
    shuffle: true
    batch_size_per_card: 128
    drop_last: true
    num_workers: 8

Eval:
  dataset:
    name: SimpleDataSet
    data_dir: {str(PADDLE_DATA_DIR / 'val').replace(chr(92), '/')}
    label_file_list:
      - {str(PADDLE_DATA_DIR / 'val' / 'label.txt').replace(chr(92), '/')}
    transforms:
      - DecodeImage:
          img_mode: BGR
          channel_first: false
      - MultiLabelEncode:
      - RecResizeImg:
          image_shape: [3, 48, 320]
      - KeepKeys:
          keep_keys:
            - image
            - label_ctc
            - label_sar
            - length
            - valid_ratio
  loader:
    shuffle: false
    drop_last: false
    batch_size_per_card: 256
    num_workers: 8
'''

config_file = PADDLE_DATA_DIR / 'rec_korean_finetune.yml'
with open(config_file, 'w', encoding='utf-8') as f:
    f.write(config_content)

print(f"설정 파일 저장: {config_file}")

설정 파일 저장: ../paddle_data/rec_korean_finetune.yml


## 5. 사전 학습 모델 다운로드

In [28]:
# 한국어 PP-OCRv3 사전 학습 모델 다운로드
import urllib.request
import tarfile

pretrain_dir = Path('pretrain_models')
pretrain_dir.mkdir(exist_ok=True)

# 한국어 인식 모델 다운로드 (PP-OCRv3)
model_url = "https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/korean_PP-OCRv3_rec_train.tar"
model_tar = pretrain_dir / "korean_PP-OCRv3_rec_train.tar"

if not (pretrain_dir / "korean_PP-OCRv3_rec_train").exists():
    print("사전 학습 모델 다운로드 중...")
    urllib.request.urlretrieve(model_url, model_tar)
    
    print("압축 해제 중...")
    with tarfile.open(model_tar, 'r') as tar:
        tar.extractall(pretrain_dir)
    
    # tar 파일 삭제
    model_tar.unlink()
    print("완료!")
else:
    print("사전 학습 모델이 이미 존재합니다.")

사전 학습 모델 다운로드 중...
압축 해제 중...


/tmp/ipykernel_1556675/930487935.py:18: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(pretrain_dir)


완료!


## 6. 모델 학습

In [30]:
import subprocess
subprocess.run([
    'python', 'tools/train.py',
    '-c', '../paddle_data/rec_korean_finetune.yml',
    '-o', 'Global.pretrained_model=null'
])

Skipping import of the encryption module.
[2026/01/29 11:21:43] ppocr INFO: Architecture : 
[2026/01/29 11:21:43] ppocr INFO:     Backbone : 
[2026/01/29 11:21:43] ppocr INFO:         last_conv_stride : [1, 2]
[2026/01/29 11:21:43] ppocr INFO:         last_pool_type : avg
[2026/01/29 11:21:43] ppocr INFO:         name : MobileNetV1Enhance
[2026/01/29 11:21:43] ppocr INFO:         scale : 0.5
[2026/01/29 11:21:43] ppocr INFO:     Head : 
[2026/01/29 11:21:43] ppocr INFO:         head_list : 
[2026/01/29 11:21:43] ppocr INFO:             CTCHead : 
[2026/01/29 11:21:43] ppocr INFO:                 Head : 
[2026/01/29 11:21:43] ppocr INFO:                     fc_decay : 1e-05
[2026/01/29 11:21:43] ppocr INFO:                 Neck : 
[2026/01/29 11:21:43] ppocr INFO:                     depth : 2
[2026/01/29 11:21:43] ppocr INFO:                     dims : 64
[2026/01/29 11:21:43] ppocr INFO:                     hidden_dims : 120
[2026/01/29 11:21:43] ppocr INFO:                     name :

W0129 11:21:43.466468 1560685 gpu_resources.cc:119] Please NOTE: device: 0, GPU Compute Capability: 7.0, Driver API Version: 12.4, Runtime API Version: 12.0
W0129 11:21:43.467262 1560685 gpu_resources.cc:164] device: 0, cuDNN Version: 9.8.


[2026/01/29 11:21:43] ppocr INFO: train dataloader has 1265 iters
[2026/01/29 11:21:43] ppocr INFO: valid dataloader has 71 iters
[2026/01/29 11:21:43] ppocr INFO: train from scratch
[2026/01/29 11:21:43] ppocr INFO: During the training process, after the 0th iteration, an evaluation is run every 500 iterations


--------------------------------------
C++ Traceback (most recent call last):
--------------------------------------
No stack trace in paddle, may be caused by external reasons.

----------------------
Error Message Summary:
----------------------
FatalError: `Segmentation fault` is detected by the operating system.
  [TimeInfo: *** Aborted at 1769653309 (unix time) try "date -d @1769653309" if you are using GNU date ***]
  [SignalInfo: *** SIGSEGV (@0x0) received by PID 1560685 (TID 0x7fced4474740) from PID 0 ***]



CompletedProcess(args=['python', 'tools/train.py', '-c', '../paddle_data/rec_korean_finetune.yml', '-o', 'Global.pretrained_model=null'], returncode=-11)

In [31]:
# 학습 실행 (GPU 사용)
!python tools/train.py -c {str(config_file).replace(chr(92), '/')}

Skipping import of the encryption module.
[2026/01/29 11:21:52] ppocr INFO: Architecture : 
[2026/01/29 11:21:52] ppocr INFO:     Backbone : 
[2026/01/29 11:21:52] ppocr INFO:         last_conv_stride : [1, 2]
[2026/01/29 11:21:52] ppocr INFO:         last_pool_type : avg
[2026/01/29 11:21:52] ppocr INFO:         name : MobileNetV1Enhance
[2026/01/29 11:21:52] ppocr INFO:         scale : 0.5
[2026/01/29 11:21:52] ppocr INFO:     Head : 
[2026/01/29 11:21:52] ppocr INFO:         head_list : 
[2026/01/29 11:21:52] ppocr INFO:             CTCHead : 
[2026/01/29 11:21:52] ppocr INFO:                 Head : 
[2026/01/29 11:21:52] ppocr INFO:                     fc_decay : 1e-05
[2026/01/29 11:21:52] ppocr INFO:                 Neck : 
[2026/01/29 11:21:52] ppocr INFO:                     depth : 2
[2026/01/29 11:21:52] ppocr INFO:                     dims : 64
[2026/01/29 11:21:52] ppocr INFO:                     hidden_dims : 120
[2026/01/29 11:21:52] ppocr INFO:                     name :

In [ ]:
# CPU로 학습 (GPU가 없는 경우)
# !python tools/train.py -c {str(config_file).replace(chr(92), '/')} -o Global.use_gpu=false

## 7. 모델 평가

In [ ]:
# 학습된 모델 평가
!python tools/eval.py -c {str(config_file).replace(chr(92), '/')} \
    -o Global.checkpoints=./output/rec_korean_finetune/best_accuracy

## 8. 추론 모델 내보내기

In [ ]:
# 추론용 모델로 변환
!python tools/export_model.py -c {str(config_file).replace(chr(92), '/')} \
    -o Global.pretrained_model=./output/rec_korean_finetune/best_accuracy \
    Global.save_inference_dir=./output/rec_korean_finetune/inference

## 9. 파인튜닝된 모델로 추론 테스트

In [ ]:
from paddleocr import PaddleOCR

# 파인튜닝된 모델 로드
ocr = PaddleOCR(
    rec_model_dir='./output/rec_korean_finetune/inference',
    rec_char_dict_path=str(dict_file),
    use_angle_cls=False,
    lang='korean'
)

# 테스트 이미지로 추론
test_image = str(BASE_DIR / 'img' / '운송장 예시파일.jpg')
result = ocr.ocr(test_image, cls=False)

print("OCR 결과:")
print("=" * 50)
for line in result[0]:
    bbox, (text, confidence) = line
    print(f"텍스트: {text}")
    print(f"신뢰도: {confidence:.4f}")
    print("-" * 30)

In [ ]:
# 생성된 이미지로 테스트
import matplotlib.pyplot as plt
from PIL import Image

test_images = list((GENERATED_DIR / 'images').glob('*.jpg'))[:5]

fig, axes = plt.subplots(1, len(test_images), figsize=(20, 4))

for i, img_path in enumerate(test_images):
    result = ocr.ocr(str(img_path), cls=False)
    
    img = Image.open(img_path)
    axes[i].imshow(img)
    axes[i].axis('off')
    
    # 인식된 텍스트 표시
    texts = [line[1][0] for line in result[0]] if result[0] else []
    axes[i].set_title('\n'.join(texts[:3]), fontsize=8)

plt.tight_layout()
plt.show()

## 10. 모델 저장 및 배포

In [ ]:
# 최종 모델을 ai 폴더로 복사
import shutil

final_model_dir = BASE_DIR / 'models' / 'paddleocr_korean_finetuned'
final_model_dir.mkdir(parents=True, exist_ok=True)

# 추론 모델 복사
inference_dir = Path('./output/rec_korean_finetune/inference')
if inference_dir.exists():
    for file in inference_dir.glob('*'):
        shutil.copy(file, final_model_dir)
    
    # 문자 사전도 복사
    shutil.copy(dict_file, final_model_dir / 'korean_dict.txt')
    
    print(f"모델 저장 완료: {final_model_dir}")
else:
    print("추론 모델을 먼저 내보내세요.")

## 완료!

파인튜닝된 모델 사용법:

```python
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    rec_model_dir='models/paddleocr_korean_finetuned',
    rec_char_dict_path='models/paddleocr_korean_finetuned/korean_dict.txt',
    use_angle_cls=False,
    lang='korean'
)

result = ocr.ocr('your_image.jpg', cls=False)
```